In [1]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [2]:
from datasets import load_dataset
import re
from tensorflow.keras.preprocessing.text import Tokenizer

# Load Dataset
ds = load_dataset("roneneldan/TinyStories")

# Take a small subset
texts = ds["train"]["text"][:500]

print("Number of Stories:", len(texts))

# Lowercase + Remove Symbols
texts = [
    re.sub(r'[^a-zA-Z\s]', '', t.lower())
    for t in texts
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

# Create Input-Output Sequences
sequences = []

for text in texts:

    tokens = tokenizer.texts_to_sequences([text])[0]

    for i in range(1, len(tokens)):

        sequences.append(
            (tokens[:i], tokens[i])
        )

print("Vocabulary Size:", vocab_size)
print("Sample Sequence:", sequences[0])

Number of Stories: 500
Vocabulary Size: 3440
Sample Sequence: ([24], 17)


In [3]:
import tensorflow as tf

embedding_dim = 32

embedding_layer = tf.keras.layers.Embedding(
    vocab_size,
    embedding_dim
)

sample_input = tf.constant([[1,2,3]])

embedded = embedding_layer(sample_input)

print("Embeddings:")
print(embedded)

print("Embedding Shape:")
print(embedded.shape)

Embeddings:
tf.Tensor(
[[[ 0.01190493  0.04713861  0.04137727  0.02611749 -0.03955568
    0.01648792  0.04065294 -0.01153078  0.04064408  0.03837521
    0.04637864 -0.04920696  0.01629343  0.01011536  0.02494041
    0.02739434 -0.02069482 -0.02809911 -0.04310289 -0.01927962
    0.02073193 -0.02041373  0.01832724  0.01425219  0.0204687
    0.03270085  0.03541083  0.00855845 -0.00491856  0.03389367
    0.00670829  0.04641602]
  [-0.01883051 -0.03401718 -0.04272328  0.00957096 -0.00480963
    0.01832571 -0.00979023  0.04703671  0.03308637  0.04079514
    0.042284    0.00241275  0.01469407 -0.03279337  0.04418609
    0.04232012  0.0496694  -0.00806034 -0.03069096  0.0497174
   -0.03184837 -0.04495275  0.03404732  0.03368641  0.0148601
    0.02553875 -0.029291   -0.03918902 -0.03732947 -0.02967225
    0.00771725  0.00106434]
  [-0.00530108 -0.04717899 -0.04985887  0.00528896 -0.03628062
    0.0327014   0.01285297 -0.01939838 -0.00449     0.03826059
    0.00304791  0.0118541  -0.02819934 -0.

In [4]:
import numpy as np

def positional_encoding(max_len, d_model):

    pos = np.arange(max_len)[:, np.newaxis]

    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2*(i//2))/np.float32(d_model)
    )

    angles = pos * angle_rates

    pe = np.zeros((max_len, d_model))

    pe[:,0::2] = np.sin(
        angles[:,0::2]
    )

    pe[:,1::2] = np.cos(
        angles[:,1::2]
    )

    return pe

pe = positional_encoding(10,32)

print(pe.shape)

(10, 32)


In [5]:
import tensorflow as tf

seq_len = 5

mask = 1 - tf.linalg.band_part(
    tf.ones((seq_len,seq_len)),
    -1,
    0
)

print(mask.numpy())

[[0. 1. 1. 1. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]


In [6]:
mha = tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=32
)

x = tf.random.normal((2,5,32))

attn_output = mha(
    x,
    x,
    attention_mask=mask
)

print(
    "Attention Output Shape:",
    attn_output.shape
)

Attention Output Shape: (2, 5, 32)


In [7]:
import tensorflow as tf

class DecoderBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        num_heads,
        d_model,
        dff
    ):
        super().__init__()

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(
                dff,
                activation='relu'
            ),
            tf.keras.layers.Dense(
                d_model
            )
        ])

        self.norm1 = tf.keras.layers.LayerNormalization()

        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self,x):

        attn_output = self.mha(x,x)

        x = self.norm1(
            x + attn_output
        )

        ffn_out = self.ffn(x)

        x = self.norm2(
            x + ffn_out
        )

        return x

In [8]:
import tensorflow as tf

inputs = tf.keras.Input(
    shape=(None,)
)

x = tf.keras.layers.Embedding(
    vocab_size,
    32
)(inputs)

decoder = DecoderBlock(
    num_heads=4,
    d_model=32,
    dff=64
)

x = decoder(x)

x = decoder(x)

x = tf.keras.layers.Lambda(
    lambda t: t[:, -1, :]
)(x)

outputs = tf.keras.layers.Dense(
    vocab_size,
    activation='softmax'
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 32)  │    110,080 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_block       │ (None, None, 32)  │     21,120 │ embedding_1[0][0… │
│ (DecoderBlock)      │                   │            │ decoder_block[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32)        │          0 │ decoder_block[1]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 3440)      │    113,520 │ lambda[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 244,720 (955.94 KB)

 Trainable params: 244,720 (955.94 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = []
y = []

for seq in sequences:

    X.append(seq[0])
    y.append(seq[1])

max_len = max(
    len(seq)
    for seq in X
)

X = pad_sequences(
    X,
    maxlen=max_len,
    padding='pre'
)

# NO to_categorical
y = np.array(y)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X,
    y,
    epochs=8,
    batch_size=32
)

index_word = {
    v:k
    for k,v in tokenizer.word_index.items()
}

def predict_next_word(text):

    text = re.sub(
        r'[^a-zA-Z\s]',
        '',
        text.lower()
    )

    seq = tokenizer.texts_to_sequences(
        [text]
    )[0]

    seq = pad_sequences(
        [seq],
        maxlen=max_len,
        padding='pre'
    )

    pred = model.predict(
        seq,
        verbose=0
    )

    next_word_id = np.argmax(
        pred[0]
    )

    return index_word.get(
        next_word_id,
        "Unknown"
    )

print(
    predict_next_word(
        "once upon a time"
    )
)

X Shape: (76729, 240)
y Shape: (76729,)
Epoch 1/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 35s 12ms/step - accuracy: 0.1537 - loss: 5.2734
Epoch 2/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2066 - loss: 4.5222
Epoch 3/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2256 - loss: 4.2473
Epoch 4/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2387 - loss: 4.0509
Epoch 5/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2489 - loss: 3.8972
Epoch 6/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2572 - loss: 3.7676
Epoch 7/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2650 - loss: 3.6622
Epoch 8/8
2398/2398 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2707 - loss: 3.5701
there


In [10]:
def generate_story(
    seed_text,
    num_words
):

    for _ in range(num_words):

        next_word = predict_next_word(
            seed_text
        )

        if next_word == "Unknown":
            break

        seed_text += " " + next_word

    return seed_text

# Example
print(
    generate_story(
        "once upon a time",
        30
    )
)

once upon a time there was a little girl named lily she was three years old and she was three years old and she was three years old and she was three years old


In [11]:
print("========== GENERATED STORIES ==========")

story1 = generate_story(
    "once upon a time",
    20
)

story2 = generate_story(
    "there was a little rabbit",
    20
)

story3 = generate_story(
    "a happy boy",
    20
)

print("\nStory 1:")
print(story1)

print("\nStory 2:")
print(story2)

print("\nStory 3:")
print(story3)

========== GENERATED STORIES ==========

Story 1:
once upon a time there was a little girl named lily she was three years old and she was three years old and she

Story 2:
there was a little rabbit and he was very fast as he was so he was so he was so he was so he was

Story 3:
a happy boy he was very fast he was very fast he was a big box and he was very deep and he


In [12]:
# Model Evaluation

loss, accuracy = model.evaluate(
    X,
    y,
    verbose=1
)

print("\n========== MODEL EVALUATION ==========")
print("Vocabulary Size :", vocab_size)
print("Number of Sequences :", len(sequences))
print("Training Samples :", len(X))
print("Loss :", loss)
print("Accuracy :", accuracy)

print("\n========== CONCLUSION ==========")
print(
    "The GPT-based Story Generation model was trained on the TinyStories dataset. "
    "The model successfully learned next-word prediction and generated children's stories from short prompts."
)

2398/2398 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.2994 - loss: 3.3074

========== MODEL EVALUATION ==========
Vocabulary Size : 3440
Number of Sequences : 76729
Training Samples : 76729
Loss : 3.307358980178833
Accuracy : 0.29937833547592163

========== CONCLUSION ==========
The GPT-based Story Generation model was trained on the TinyStories dataset. The model successfully learned next-word prediction and generated children's stories from short prompts.
